# 05 – Gap-filling of the monthly snow products: steps 2 to 7

| Step | Rule |
|---|---|
| 2 | 100 m elevation bands ≤ 600 m (DJF) or ≤ 1000 m (other seasons) with < 1 % of snow: snow 1 → 0 (false snow at low elevation) |
| 3 | cloud and summer snow mask = 1 → snow (1) |
| 4 | cloud, probability field of the month = 1 and elevation > min(elevation of field = 1) + 500 m → snow (100); cloud, field = 2 and elevation < max(elevation of field = 2) − 500 m → no snow (200) |
| 5 | snow line, if cloud ≤ 40 % and snow ≥ 1 %: clouds below HSmin → 200, clouds above HSmax → 100. Percentages per band computed without lakes. HSmin band: > 0.1 % of snow (> 1 % in JJA for the eastern half, as in the original products) |
| 6 | repeated until no change: cloud with ≥ 3 of its 4 neighbours snow → 100, ≥ 3 of 4 no snow → 200 |
| 7 | repeated until no change: cloud with one of its 8 neighbours snow and lower → 100 |

**Input:** binary monthly product `Snow_<year>_<month>.tif` of `03_snow_GLMM_monthly.ipynb` (0 no snow, 1 snow, 2 cloud, 255 NA).

**Output:** `Snow_gapfilled_<year>_<month>.tif` with the codes 0, 1, 2, 100 (filled snow), 200 (filled no snow) and 255 (NA). The statistics of each step are written to an Excel file (% cloud / snow / no snow before and after, cloud reduction, HSmin, HSmax…).

**Implementation notes:**
- No intermediate files between the steps.
- Steps 6 and 7 only re-examine, at each iteration, the clouds next to the pixels changed in the previous one. The result is identical to re-scanning the whole raster, and the computation takes a few seconds for the whole of Switzerland.
- **Zones:** as in the original products, steps 2 and 4 to 7 are computed separately for the western (a) and eastern (b) halves of the grid (`zones = 'halves'`); the first and last rows / columns of each half are not filled by steps 6 and 7. With `zones = 'CH'`, the whole of Switzerland is processed at once.

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors

sys.path.insert(0, '../utils')
import snow_gapfill as gf

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Configuration section

in_dir  = '../outputs/Snow_monthly_GLMM'                 # binary products Snow_<year>_<month>.tif
out_dir = '../outputs/Snow_monthly_gapfilled'
stats_xlsx = '%s/Snow_gapfilling_stats.xlsx' % out_dir

# auxiliary data (None = step skipped)
summer_mask_tif = '../data/aux/summer_snow_mask.tif'   # step 3: 1 = snow every summer
proba_fields_pattern = '../data/aux/probability_field_%02d.tif'   # step 4: from 04_monthly_probability_fields.ipynb (1 = snow, 2 = no snow)
dem_tif = '../data/DEM/DEM_final_corrected.tif'   # from Zenodo: DEM (m, grid 14234 x 6200), ~15,500 wrong pixels (> 600 m from swissALTI3D) corrected
dem_scale = 1.0

steps = [2, 3, 4, 5, 6, 7]                      # steps to apply, in this order
step2_max_alt = {'DJF': 600, 'MAM': 1000, 'JJA': 1000, 'SON': 1000}   # step 2 (600 m checked for DJF)
step3_value = 1                                 # value of the pixels filled by the summer mask (1 as in the original products)
# step 5: minimum % of snow for a 100 m band to define HSmin (0.1 %; the original JJA products of the eastern half (b)
# used 1 %: checked on the original statistics 1985-2021, 85/85 months identical with 1 %, 90/90 with 0.1 % for a)
step5_snow_pct = {('JJA', 'b'): 1.0}            # (season, zone) -> %, default 0.1
season_of_month = {12: 'DJF', 1: 'DJF', 2: 'DJF', 3: 'MAM', 4: 'MAM', 5: 'MAM', 6: 'JJA', 7: 'JJA', 8: 'JJA', 9: 'SON', 10: 'SON', 11: 'SON'}
zones = 'halves'                                # 'halves' = west (a) / east (b) as the original products, 'CH' = whole Switzerland
step4_use_land = True                           # step 4: also apply the no-snow rule (mask == 2)

start_year, end_year = 1984, 2026
start_month, end_month = 1, 12

# NDSI grid
grid_transform = from_origin(5.699957268017272, 47.85026882419969, 0.0003407435000010325, 0.00034046861818428666)
W, H = 14234, 6199
grid_crs = 'EPSG:4326'
MASK = rasterio.open('../data/aux/mask_switzerland_without_lakes.tif').read(1) == 1
os.makedirs(out_dir, exist_ok=True)

## 1. Auxiliary data on the NDSI grid

- **DEM:** `DEM_final` of the original products (grid 14234 × 6200), resampled with nearest neighbour; 0 values become NaN. About 15,500 wrong pixels (holes filled with 0–230 m in the Alps, or values too high), differing by more than 600 m from swissALTI3D, are replaced by swissALTI3D (`DEM_final_corrected.tif`, Zenodo).
- **DEM100:** 100 m bands, floor(DEM / 100) × 100, used by steps 2, 5 and 7.
- The percentages per band are computed on all the pixels with an elevation in step 2, and without lakes in step 5. Both choices were checked against the original products.
- **Summer snow mask and probability fields:** resampled with nearest neighbour, since they are classes.

In [ ]:
def to_grid(path, resampling=Resampling.nearest, scale=1.0, cache=None):
    """One band raster on the NDSI grid (float32, NaN = no data); cached as GeoTIFF."""
    if cache and os.path.exists(cache):
        a = rasterio.open(cache).read(1)
        return np.where(np.isnan(a), np.nan, a)
    a = np.full((H, W), np.nan, 'float32')
    with rasterio.open(path) as src:
        reproject(rasterio.band(src, 1), a, src_transform=src.transform, src_crs=src.crs,
                  dst_transform=grid_transform, dst_crs=grid_crs, src_nodata=src.nodata,
                  dst_nodata=np.nan, resampling=resampling)
    a *= scale
    if cache:
        with rasterio.open(cache, 'w', driver='GTiff', width=W, height=H, count=1, dtype='float32', crs=grid_crs,
                           transform=grid_transform, nodata=np.nan, compress='deflate') as dst:
            dst.write(a, 1)
    return a

DEM = to_grid(dem_tif, Resampling.nearest, dem_scale, cache='../outputs/dem_final_corrected_ndsi_grid.tif')
DEM[DEM <= 0] = np.nan
DEM100 = np.floor(DEM / 100) * 100
SUMMER = to_grid(summer_mask_tif) if summer_mask_tif else None
print('DEM %.0f - %.0f m, %i bands of 100 m' % (np.nanmin(DEM), np.nanmax(DEM), len(np.unique(DEM100[np.isfinite(DEM100)]))))
print('summer mask:', 'yes' if SUMMER is not None else 'NO -> step 3 skipped')
print('monthly probability fields:', proba_fields_pattern or 'NO -> step 4 skipped')

# zones on which HSmin / HSmax and the step 4 limits are computed
ZONES = [slice(0, W // 2), slice(W // 2, W)] if zones == 'halves' else [slice(0, W)]

## 2. The steps for one month

In [ ]:
def gapfill_month(snow, month):
    """Apply the selected steps to one binary monthly product; returns the result and the statistics."""
    rows, cur = [], snow
    for step in steps:
        t0 = time.time()
        if step == 2:
            new, info = cur.copy(), {}
            for k, z in enumerate(ZONES):
                new[:, z], inf = gf.step2_low_snow(cur[:, z], DEM100[:, z], step2_max_alt[season_of_month[month]])
                info.update({'%s zone %i' % (key, k + 1): v for key, v in inf.items()})
        elif step == 3:
            if SUMMER is None:
                continue
            new, info = gf.step3_summer_mask(cur, SUMMER, step3_value)
        elif step == 4:
            if not proba_fields_pattern:
                continue
            field = to_grid(proba_fields_pattern % month)
            new, info = cur.copy(), {}
            for k, z in enumerate(ZONES):
                new[:, z], inf = gf.step4_monthly_proba(cur[:, z], field[:, z], DEM[:, z], step4_use_land)
                info.update({'%s zone %i' % (key, k + 1): v for key, v in inf.items()})
        elif step == 5:
            new, info = cur.copy(), {}
            for k, z in enumerate(ZONES):
                zname = 'ab'[k] if len(ZONES) == 2 else 'CH'
                pct = step5_snow_pct.get((season_of_month[month], zname), 0.1)
                new[:, z], inf = gf.step5_snowline(cur[:, z], DEM100[:, z], snow_band_pct=pct)
                info.update({'%s zone %i' % (key, k + 1): v for key, v in inf.items()})
        elif step == 6:                              # per zone, borders not filled (as the original notebooks)
            new, info = cur.copy(), {}
            for k, z in enumerate(ZONES):
                new[:, z], inf = gf.step6_4nb(cur[:, z])
                info['Iterations zone %i' % (k + 1)] = inf['Iterations']
        elif step == 7:
            new, info = cur.copy(), {}
            for k, z in enumerate(ZONES):
                new[:, z], inf = gf.step7_8nb(cur[:, z], DEM100[:, z])
                info['Iterations zone %i' % (k + 1)] = inf['Iterations']
        info['time (s)'] = round(time.time() - t0, 1)
        rows.append(gf.stats_row('Step %i' % step, cur, new, info))
        cur = new
    return cur, rows


def write_tif(path, arr):
    with rasterio.open(path, 'w', driver='GTiff', width=W, height=H, count=1, dtype='uint8', crs=grid_crs,
                       transform=grid_transform, nodata=255, compress='deflate', tiled=True) as dst:
        dst.write(arr, 1)

cmap = mcolors.ListedColormap(['#8c6d46', '#3a7bd5', '#cccccc', '#9ec9f5', '#d9c3a0'])
norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 100.5, 200.5], cmap.N)     # 0, 1, 2, 100, 200
def show(arr, title, ax):
    ax.imshow(np.where(arr == 255, np.nan, arr)[::4, ::4], cmap=cmap, norm=norm, interpolation='nearest')
    ax.set_title(title); ax.axis('off')

## 3. Test on one month

In [ ]:
test_year, test_month = 2022, 11
f = '%s/Snow_%i_%02d.tif' % (in_dir, test_year, test_month)
snow = rasterio.open(f).read(1)
res, rows = gapfill_month(snow, test_month)
df_test = pd.DataFrame(rows)
pd.set_option('display.max_columns', 40)
df_test

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(22, 6))
show(snow, 'binary product  -  cloud %.1f %%' % gf.percentages(snow)[0], ax[0])
show(res, 'after %s  -  cloud %.1f %%' % (', '.join(r['Step'] for r in rows), gf.percentages(res)[0]), ax[1])
plt.tight_layout()

## 4. All months

- Months already computed are skipped.
- The statistics are added to the Excel file: one row per month and step.

In [ ]:
stats = pd.read_excel(stats_xlsx).to_dict('records') if os.path.exists(stats_xlsx) else []

for year in range(start_year, end_year + 1):
    for month in range(start_month, end_month + 1):
        f_in = '%s/Snow_%i_%02d.tif' % (in_dir, year, month)
        f_out = '%s/Snow_gapfilled_%i_%02d.tif' % (out_dir, year, month)
        if not os.path.exists(f_in) or os.path.exists(f_out):
            continue
        t0 = time.time()
        res, rows = gapfill_month(rasterio.open(f_in).read(1), month)
        write_tif(f_out, res)
        date = '%i-%02d' % (year, month)
        stats = [r for r in stats if r['Date'] != date] + [dict(Date=date, **r) for r in rows]
        pd.DataFrame(stats).sort_values(['Date', 'Step']).to_excel(stats_xlsx, index=False)
        print('%s: cloud %.1f %% -> %.1f %% (%.0f s)' % (date, rows[0]['Cloud Percent Before'], rows[-1]['Cloud Percent After'], time.time() - t0))

print('ALL DONE')